# 00 Setup (run once)
Creates the schemas and tables the pipeline needs, and seeds the watermark row.

In [ ]:
for schema in ["bronze", "silver", "gold", "staging", "metadata"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

spark.sql("""
CREATE TABLE IF NOT EXISTS metadata.pipeline_metadata (
    pipeline_name STRING,
    watermark TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS silver.sales (
    customer_id INT,
    amount DOUBLE,
    region STRING,
    updated_at TIMESTAMP
) USING DELTA
""")

# Seed the watermark at the epoch so the first run picks up everything.
spark.sql("""
MERGE INTO metadata.pipeline_metadata m
USING (SELECT 'sales_pipeline' AS pipeline_name, TIMESTAMP'1970-01-01 00:00:00' AS watermark) s
ON m.pipeline_name = s.pipeline_name
WHEN NOT MATCHED THEN INSERT *
""")